# Detect bottle-fall events around Johnnie Walker Black Label stoppages

This notebook uses the included Johnnie Walker Black Label PLC events to define tightly controlled S3 search windows.

For each stoppage it:

1. Converts the factory-local event time to UTC.
2. Looks from `N` minutes before to `N` minutes after the stop (`N=1` initially).
3. Uses filename timestamps plus probed video durations to allowlist only clips overlapping that window.
4. Downloads every allowlisted clip and rotates videos dated from 21 June 2026 onward by 180Ã‚Â° before classification.
5. Saves every video locally under `classified_videos/<predicted_class>/`.
6. Saves one annotated peak-anomaly frame from every video.
7. Scans classified frames chronologically and assigns the stoppage using the first non-normal detection.
8. Extracts the first non-normal frame and estimates its exact timestamp as `filename clip start + frame time`.

The CNN classes remain `normal`, `fallen_before_entry`, and `fallen_in_view`. Each anomaly class keeps its own configured threshold and is scored over complete rolling windows.

### Performance limitation

Every CSV row is a positive stoppage example. The notebook can measure stoppage recall and false negatives, but precision, specificity and F1 still require separate normal/control event windows.

---

## 1. Imports and configuration

Set `run_s3_batch=True` when ready to download and classify. Use `max_clips=5` for the first test run.

In [ ]:
import bisect
import csv
import hashlib
import json
import shutil
import subprocess
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass
from datetime import datetime, timedelta
from pathlib import Path
from typing import Iterator

import boto3
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from botocore.exceptions import BotoCoreError, ClientError, TokenRetrievalError
from IPython.display import display
from PIL import Image, ImageOps
from torch.utils.data import DataLoader, Dataset
from torchvision.models import resnet50
from torchvision.transforms import v2 as transforms

from VideoModule.io.clip_export import ffprobe_path

# Find the repository root whether Jupyter starts in the repo or a subfolder.
working_directory = Path.cwd().resolve()
repository_candidates = [working_directory, *working_directory.parents]
REPO_ROOT = next(
    candidate
    for candidate in repository_candidates
    if (candidate / "VideoModule").is_dir()
)
PIPELINE_ROOT = REPO_ROOT / "stoppage_detection_and_classification"
CNN_CLASSIFIER_DIR = PIPELINE_ROOT / "cnn_classifier"
PLC_EVENTS_DIR = PIPELINE_ROOT / "plc_stoppage_events"
PLC_L14_EVENTS_DIR = PLC_EVENTS_DIR / "output" / "01_prepare_events"

DEFAULT_MODEL_PATH = CNN_CLASSIFIER_DIR / "output" / "02_train_cnn" / "best_model.pth"
DEFAULT_METADATA_PATH = CNN_CLASSIFIER_DIR / "output" / "02_train_cnn" / "model_metadata.json"
DEFAULT_OUTPUT_DIR = CNN_CLASSIFIER_DIR / "output" / "03_classify_s3_clips"
DEFAULT_AWS_PROFILE = "DashcamGlbDiageoProdDataContrib-522196013725"
DEFAULT_S3_BUCKET = "diageo-prod-global-dashcam-mc-nuc-video"
DEFAULT_S3_PREFIX = "cortexvpu-01a-005-41884872/"
DEFAULT_EVENT_TIMEZONE = "Europe/London"
VIDEO_EXTENSIONS = {".ts", ".mp4", ".avi", ".mov", ".mkv", ".m4v"}
ANOMALY_CLASSES = {"fallen_before_entry", "fallen_in_view"}
TEMPORAL_AGGREGATION_VERSION = "class-specific-temporal-v3-no-bottle"
MODEL_HANDOFF_SCHEMA_VERSION = 3
IMAGE_PREPROCESSING_VERSION = "aspect-preserving-letterbox-v1"


def normalize_optional_utc_timestamp(value: str | None) -> datetime | None:
    """Parse an optional datetime string into the naive UTC convention."""
    if value is None:
        return None

    timestamp = pd.Timestamp(value)
    if timestamp.tzinfo is not None:
        timestamp = timestamp.tz_convert("UTC").tz_localize(None)
    return timestamp.to_pydatetime()


@dataclass(frozen=True)
class Config:
    """Settings for matching, classification and evaluation."""

    # Input and output paths.
    events_csv: Path | None = None  # None selects the newest dated extract.
    model_path: Path = DEFAULT_MODEL_PATH
    metadata_path: Path = DEFAULT_METADATA_PATH
    output_dir: Path = DEFAULT_OUTPUT_DIR

    # S3 source and event-time interpretation.
    aws_profile: str = DEFAULT_AWS_PROFILE
    s3_bucket: str = DEFAULT_S3_BUCKET
    s3_prefix: str = DEFAULT_S3_PREFIX
    # Inclusive UTC bounds, for example "2026-07-01 04:30:00".
    # Leave either string as None when that side should be unbounded.
    s3_start_timestamp_utc: str | None = "2026-07-02 00:00:00"
    s3_end_timestamp_utc: str | None = None
    event_timezone: str = DEFAULT_EVENT_TIMEZONE
    match_tolerance_seconds: float = 30.0
    event_window_minutes: float = 0.5  # Search 30 seconds before and after each event.
    listing_padding_seconds: float = 30.0
    duration_probe_timeout_seconds: float = 120.0
    duration_probe_workers: int = 4

    # Inference. None classifies every frame; use 10.0 for 10 FPS.
    sample_fps: float | None = 60.0
    rotate_from_timestamp_utc: datetime = datetime(2026, 6, 21, 0, 0, 0)
    batch_size: int = 64
    rolling_window_seconds: float = 0.5
    persistent_seconds: float = 0.3
    min_falling_seconds: float = 0.10
    peak_support_seconds: float = 0.10
    normal_clear_seconds: float = 0.5
    fallen_clear_seconds: float = 0.75
    normal_threshold: float = 0.75
    fault_activity_threshold: float = 0.50
    falling_threshold: float = 0.60
    falling_peak_threshold: float = 0.80
    fallen_threshold: float = 0.65

    # Run controls.
    run_s3_batch: bool = True
    max_clips: int | None = None
    reprocess: bool = True


CFG = Config()
S3_START_TIMESTAMP_UTC = normalize_optional_utc_timestamp(
    CFG.s3_start_timestamp_utc
)
S3_END_TIMESTAMP_UTC = normalize_optional_utc_timestamp(
    CFG.s3_end_timestamp_utc
)

if (
    S3_START_TIMESTAMP_UTC is not None
    and S3_END_TIMESTAMP_UTC is not None
    and S3_START_TIMESTAMP_UTC > S3_END_TIMESTAMP_UTC
):
    raise ValueError(
        "s3_start_timestamp_utc must be before or equal to "
        "s3_end_timestamp_utc"
    )

if (
    CFG.match_tolerance_seconds < 0
    or CFG.event_window_minutes <= 0
    or CFG.listing_padding_seconds <= 0
):
    raise ValueError("Timestamp tolerances must be valid")
if CFG.duration_probe_timeout_seconds <= 0 or CFG.duration_probe_workers < 1:
    raise ValueError("Duration probe settings must be positive")
if CFG.sample_fps is not None and CFG.sample_fps <= 0:
    raise ValueError("sample_fps must be None or positive")
if CFG.batch_size < 1 or CFG.rolling_window_seconds <= 0:
    raise ValueError("batch_size and rolling_window_seconds must be positive")
for duration_name in (
    "persistent_seconds",
    "min_falling_seconds",
    "peak_support_seconds",
    "normal_clear_seconds",
    "fallen_clear_seconds",
):
    if getattr(CFG, duration_name) <= 0:
        raise ValueError(f"{duration_name} must be positive")
for threshold_name in (
    "normal_threshold",
    "fault_activity_threshold",
    "falling_threshold",
    "falling_peak_threshold",
    "fallen_threshold",
):
    threshold = getattr(CFG, threshold_name)
    if not 0.0 <= threshold <= 1.0:
        raise ValueError(f"{threshold_name} must be between 0 and 1")

CFG.output_dir.mkdir(parents=True, exist_ok=True)
print(f"Repository:       {REPO_ROOT}")
print(f"Output:           {CFG.output_dir}")
print(f"Frame sampling:   {'every frame' if CFG.sample_fps is None else f'{CFG.sample_fps:g} FPS'}")
print(
    "Rotate 180 degrees: camera timestamps from "
    f"{CFG.rotate_from_timestamp_utc.isoformat()} UTC onward"
)
if S3_START_TIMESTAMP_UTC is None and S3_END_TIMESTAMP_UTC is None:
    s3_timestamp_window_text = "all timestamps"
else:
    s3_start_text = (
        S3_START_TIMESTAMP_UTC.isoformat()
        if S3_START_TIMESTAMP_UTC is not None
        else "unbounded"
    )
    s3_end_text = (
        S3_END_TIMESTAMP_UTC.isoformat()
        if S3_END_TIMESTAMP_UTC is not None
        else "unbounded"
    )
    s3_timestamp_window_text = f"{s3_start_text} to {s3_end_text} UTC"
print(f"S3 timestamp window: {s3_timestamp_window_text}")
print(f"Stop window:      {CFG.event_window_minutes:g} minute(s) each side")
print(f"CNN window:       {CFG.rolling_window_seconds:g} seconds")
print(f"Normal threshold:  {CFG.normal_threshold:.2f}")
print(f"Falling threshold: {CFG.falling_threshold:.2f}")
print(f"Falling peak:      {CFG.falling_peak_threshold:.2f}")
print(f"Fallen threshold:  {CFG.fallen_threshold:.2f}")
print(f"Run S3 batch:     {CFG.run_s3_batch}")

---

## 2. Event extract and S3 timestamp matching helpers

In [ ]:
@dataclass(frozen=True)
class S3Video:
    """A timestamped video object found in S3."""

    bucket: str
    key: str
    timestamp_utc: datetime


def find_latest_event_extract() -> Path:
    """Return the newest dated Johnnie Walker event extract."""

    candidates = sorted(
        PLC_L14_EVENTS_DIR.glob("johnnie_walker_black_label_events_*.csv"),
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )

    if not candidates:
        raise FileNotFoundError(
            "No johnnie_walker_black_label_events_*.csv extract was found in "
            f"{PLC_L14_EVENTS_DIR}"
        )

    return candidates[0]


def load_events(events_csv: Path, event_timezone: str) -> pd.DataFrame:
    """Load included Johnnie Walker events and add stable UTC timestamps."""

    events = pd.read_csv(events_csv)

    if "Event Time" not in events.columns:
        raise ValueError(f"{events_csv} does not contain an 'Event Time' column.")

    # Keep only included records if the extract contains an inclusion column.
    if "Include_Status" in events.columns:
        include_mask = (
            events["Include_Status"].astype(str).str.strip().str.casefold()
            == "include"
        )
        events = events.loc[include_mask].copy()

    # Validate the product when the extract contains a product description.
    if "Product" in events.columns:
        product_mask = (
            events["Product"]
            .astype(str)
            .str.contains(
                r"(?:johnnie\s+walker|jw)\s+black",
                case=False,
                na=False,
                regex=True,
            )
        )
        events = events.loc[product_mask].copy()

    if events.empty:
        raise ValueError("The event extract contains no included Johnnie Walker events.")

    # Treat source event times as factory-local times, then convert them to UTC.
    local_times = pd.to_datetime(events["Event Time"], errors="raise")
    local_times = local_times.dt.tz_localize(
        event_timezone,
        ambiguous="raise",
        nonexistent="raise",
    )
    utc_times = local_times.dt.tz_convert("UTC")

    events = events.reset_index(drop=True)
    events.insert(
        0,
        "event_id",
        [f"event_{index + 1:06d}" for index in range(len(events))],
    )
    events["event_time_local"] = local_times.reset_index(drop=True)
    events["event_time_utc"] = utc_times.reset_index(drop=True)

    return events


def create_s3_client(profile_name: str):
    """Create an S3 client for the configured AWS profile."""

    try:
        session = boto3.Session(profile_name=profile_name)
        return session.client("s3")
    except (BotoCoreError, ClientError, TokenRetrievalError) as error:
        raise RuntimeError(
            "AWS authentication failed. Run:\n"
            f"aws sso login --profile {profile_name}"
        ) from error


def parse_timestamp_from_s3_key(key: str) -> datetime | None:
    """Read the UTC clip timestamp from the standard camera filename."""

    filename = Path(key).name

    # Locate YYYY-MM-DD_HH-MM-SS_microseconds within the filename.
    parts = filename.split("_")
    for index in range(len(parts) - 2):
        candidate = "_".join(parts[index : index + 3])
        candidate = candidate.split(".")[0]

        try:
            return datetime.strptime(candidate, "%Y-%m-%d_%H-%M-%S_%f")
        except ValueError:
            continue

    return None


def list_s3_videos(
    s3_client,
    bucket: str,
    prefix: str,
    start_utc: datetime | None = None,
    end_utc: datetime | None = None,
) -> list[S3Video]:
    """List S3 videos whose starts fall inside optional inclusive UTC bounds."""

    videos: list[S3Video] = []
    paginator = s3_client.get_paginator("list_objects_v2")

    try:
        pages = paginator.paginate(Bucket=bucket, Prefix=prefix)
        for page in pages:
            for item in page.get("Contents", []):
                key = item["Key"]

                if Path(key).suffix.lower() not in VIDEO_EXTENSIONS:
                    continue

                timestamp_utc = parse_timestamp_from_s3_key(key)
                if timestamp_utc is None:
                    continue

                if start_utc is not None and timestamp_utc < start_utc:
                    continue
                if end_utc is not None and timestamp_utc > end_utc:
                    continue

                videos.append(
                    S3Video(
                        bucket=bucket,
                        key=key,
                        timestamp_utc=timestamp_utc,
                    )
                )
    except (BotoCoreError, ClientError, TokenRetrievalError) as error:
        raise RuntimeError(
            "Could not list S3 videos. Your AWS SSO session may have expired. "
            "Run:\n"
            f"aws sso login --profile {DEFAULT_AWS_PROFILE}"
        ) from error

    videos.sort(key=lambda video: video.timestamp_utc)
    return videos


def find_two_closest_videos(
    event_time_utc: datetime,
    videos: list[S3Video],
    video_timestamps: list[datetime],
) -> list[S3Video]:
    """Return the two filename timestamps closest to an event."""

    if not videos:
        return []

    # In a sorted timestamp list, the two absolute-nearest values must be
    # among the two values on either side of the insertion position.
    insertion_index = bisect.bisect_left(video_timestamps, event_time_utc)
    search_start = max(0, insertion_index - 2)
    search_stop = min(len(videos), insertion_index + 2)
    nearby_videos = videos[search_start:search_stop]

    ordered_candidates = sorted(
        nearby_videos,
        key=lambda video: (
            abs((video.timestamp_utc - event_time_utc).total_seconds()),
            video.timestamp_utc,
        ),
    )
    return ordered_candidates[:2]


def candidate_videos_for_event_window(
    event_time_utc: datetime,
    videos: list[S3Video],
    video_timestamps: list[datetime],
    event_window_seconds: float,
) -> list[S3Video]:
    """Return starts inside the window plus one possible overlapping predecessor."""

    window_start = event_time_utc - timedelta(seconds=event_window_seconds)
    window_end = event_time_utc + timedelta(seconds=event_window_seconds)
    first_inside = bisect.bisect_left(video_timestamps, window_start)
    after_window = bisect.bisect_right(video_timestamps, window_end)
    candidate_start = max(0, first_inside - 1)
    return videos[candidate_start:after_window]


def collect_duration_candidates(
    events: pd.DataFrame,
    videos: list[S3Video],
    event_window_seconds: float,
) -> list[S3Video]:
    """Collect unique clips needed for two-closest and event-window matching."""

    video_timestamps = [video.timestamp_utc for video in videos]
    candidates_by_key = {}

    for _, event in events.iterrows():
        event_time_utc = event["event_time_utc"].to_pydatetime().replace(tzinfo=None)

        # Keep the two filename timestamps required for anchor matching.
        for video in find_two_closest_videos(
            event_time_utc,
            videos,
            video_timestamps,
        ):
            candidates_by_key[video.key] = video

        # Add starts in the configured event window and one predecessor
        # whose duration may make it overlap the beginning of the window.
        for video in candidate_videos_for_event_window(
            event_time_utc,
            videos,
            video_timestamps,
            event_window_seconds,
        ):
            candidates_by_key[video.key] = video

    return sorted(
        candidates_by_key.values(),
        key=lambda video: video.timestamp_utc,
    )


def probe_s3_video_duration_seconds(
    s3_client,
    video: S3Video,
    timeout_seconds: float,
) -> float:
    """Probe duration through a temporary presigned S3 URL without saving the clip."""

    executable = ffprobe_path()
    if executable is None:
        raise RuntimeError("ffprobe is required to resolve S3 clip durations")

    presigned_url = s3_client.generate_presigned_url(
        "get_object",
        Params={"Bucket": video.bucket, "Key": video.key},
        ExpiresIn=3600,
    )
    command = [
        executable,
        "-v",
        "error",
        "-show_entries",
        "format=duration:stream=duration",
        "-of",
        "json",
        presigned_url,
    ]
    completed = subprocess.run(
        command,
        capture_output=True,
        text=True,
        timeout=timeout_seconds,
        check=False,
    )
    if completed.returncode != 0:
        error_message = completed.stderr.strip()[-500:]
        raise RuntimeError(
            f"ffprobe failed for s3://{video.bucket}/{video.key}: {error_message}"
        )

    probe_result = json.loads(completed.stdout)
    duration_values = []

    format_duration = probe_result.get("format", {}).get("duration")
    if format_duration not in (None, "N/A"):
        duration_values.append(float(format_duration))

    for stream in probe_result.get("streams", []):
        stream_duration = stream.get("duration")
        if stream_duration not in (None, "N/A"):
            duration_values.append(float(stream_duration))

    valid_durations = [
        duration for duration in duration_values
        if np.isfinite(duration) and duration > 0
    ]
    if not valid_durations:
        raise RuntimeError(f"No valid duration returned for {video.key}")

    return max(valid_durations)


def load_or_probe_s3_durations(
    s3_client,
    candidate_videos: list[S3Video],
    cache_path: Path,
    timeout_seconds: float,
    max_workers: int,
) -> tuple[dict[str, float], pd.DataFrame]:
    """Load cached durations and concurrently probe only missing candidates."""

    cached_rows = []
    duration_by_key = {}
    if cache_path.exists():
        cached_frame = pd.read_csv(cache_path)
        cached_rows = cached_frame.to_dict(orient="records")
        successful_cache = cached_frame.loc[
            cached_frame["status"].astype(str).str.casefold() == "success"
        ]
        duration_by_key.update(
            dict(zip(
                successful_cache["s3_key"].astype(str),
                pd.to_numeric(successful_cache["duration_seconds"]),
            ))
        )

    missing_videos = [
        video for video in candidate_videos
        if video.key not in duration_by_key
    ]
    print(
        f"Duration candidates: {len(candidate_videos):,}; "
        f"cached: {len(candidate_videos) - len(missing_videos):,}; "
        f"to probe: {len(missing_videos):,}"
    )

    new_rows = []
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_video = {
            executor.submit(
                probe_s3_video_duration_seconds,
                s3_client,
                video,
                timeout_seconds,
            ): video
            for video in missing_videos
        }
        for completed_count, future in enumerate(
            as_completed(future_to_video),
            start=1,
        ):
            video = future_to_video[future]
            try:
                duration_seconds = float(future.result())
                duration_by_key[video.key] = duration_seconds
                status = "success"
                error = ""
            except Exception as exception:
                duration_seconds = np.nan
                status = "failed"
                error = str(exception)

            new_rows.append({
                "s3_bucket": video.bucket,
                "s3_key": video.key,
                "clip_timestamp_utc": video.timestamp_utc.isoformat(),
                "duration_seconds": duration_seconds,
                "status": status,
                "error": error,
                "probed_at_utc": pd.Timestamp.now(tz="UTC").isoformat(),
            })
            if completed_count % 20 == 0 or completed_count == len(missing_videos):
                print(f"  Probed {completed_count:,}/{len(missing_videos):,} durations")

    # Keep the newest result for each key so failed probes can be retried later.
    duration_cache = pd.DataFrame(cached_rows + new_rows)
    if not duration_cache.empty:
        duration_cache = duration_cache.drop_duplicates("s3_key", keep="last")
        duration_cache = duration_cache.sort_values("clip_timestamp_utc")
        cache_path.parent.mkdir(parents=True, exist_ok=True)
        duration_cache.to_csv(cache_path, index=False)

    return duration_by_key, duration_cache


def match_events_to_videos(
    events: pd.DataFrame,
    videos: list[S3Video],
    duration_by_key: dict[str, float],
    fallback_tolerance_seconds: float,
) -> pd.DataFrame:
    """Select the candidate whose timestamp-plus-duration contains each event."""

    video_timestamps = [video.timestamp_utc for video in videos]
    matches = []

    for _, event in events.iterrows():
        event_time_utc = event["event_time_utc"].to_pydatetime().replace(tzinfo=None)
        candidates = find_two_closest_videos(
            event_time_utc,
            videos,
            video_timestamps,
        )

        candidate_details = []
        for video in candidates:
            duration_seconds = duration_by_key.get(video.key)
            clip_end_utc = (
                video.timestamp_utc + timedelta(seconds=duration_seconds)
                if duration_seconds is not None
                else None
            )
            contains_event = bool(
                clip_end_utc is not None
                and video.timestamp_utc <= event_time_utc <= clip_end_utc
            )
            candidate_details.append({
                "video": video,
                "duration_seconds": duration_seconds,
                "clip_end_utc": clip_end_utc,
                "contains_event": contains_event,
                "absolute_start_difference_seconds": abs(
                    (video.timestamp_utc - event_time_utc).total_seconds()
                ),
                "event_offset_seconds": (
                    event_time_utc - video.timestamp_utc
                ).total_seconds(),
            })

        exact_start_candidates = [
            detail for detail in candidate_details
            if detail["absolute_start_difference_seconds"] < 1e-6
        ]
        containing_candidates = [
            detail for detail in candidate_details
            if detail["contains_event"]
        ]

        selected_detail = None
        selection_method = "unmatched"
        if exact_start_candidates:
            selected_detail = exact_start_candidates[0]
            selection_method = "exact_clip_start"
        elif containing_candidates:
            # If intervals overlap, prefer the clip that started most recently.
            selected_detail = max(
                containing_candidates,
                key=lambda detail: detail["video"].timestamp_utc,
            )
            selection_method = "event_inside_clip_interval"
        elif candidate_details:
            nearest_detail = min(
                candidate_details,
                key=lambda detail: detail["absolute_start_difference_seconds"],
            )
            if (
                nearest_detail["absolute_start_difference_seconds"]
                <= fallback_tolerance_seconds
            ):
                selected_detail = nearest_detail
                selection_method = "nearest_start_fallback"

        matched = selected_detail is not None
        match_row = event.to_dict()
        match_row["matched"] = matched
        match_row["selection_method"] = selection_method

        for candidate_number in (1, 2):
            prefix = f"candidate_{candidate_number}"
            if candidate_number <= len(candidate_details):
                detail = candidate_details[candidate_number - 1]
                match_row[f"{prefix}_s3_key"] = detail["video"].key
                match_row[f"{prefix}_clip_start_utc"] = detail[
                    "video"
                ].timestamp_utc.isoformat()
                match_row[f"{prefix}_duration_seconds"] = detail[
                    "duration_seconds"
                ]
                match_row[f"{prefix}_clip_end_utc"] = (
                    detail["clip_end_utc"].isoformat()
                    if detail["clip_end_utc"] is not None
                    else ""
                )
                match_row[f"{prefix}_contains_event"] = detail[
                    "contains_event"
                ]
                match_row[f"{prefix}_start_difference_seconds"] = detail[
                    "absolute_start_difference_seconds"
                ]
            else:
                match_row[f"{prefix}_s3_key"] = ""
                match_row[f"{prefix}_clip_start_utc"] = ""
                match_row[f"{prefix}_duration_seconds"] = np.nan
                match_row[f"{prefix}_clip_end_utc"] = ""
                match_row[f"{prefix}_contains_event"] = False
                match_row[f"{prefix}_start_difference_seconds"] = np.nan

        if matched:
            selected_video = selected_detail["video"]
            match_row["s3_bucket"] = selected_video.bucket
            match_row["s3_key"] = selected_video.key
            match_row["clip_timestamp_utc"] = selected_video.timestamp_utc.isoformat()
            match_row["clip_duration_seconds"] = selected_detail[
                "duration_seconds"
            ]
            match_row["clip_end_utc"] = (
                selected_detail["clip_end_utc"].isoformat()
                if selected_detail["clip_end_utc"] is not None
                else ""
            )
            match_row["event_offset_seconds"] = selected_detail[
                "event_offset_seconds"
            ]
            match_row["event_to_clip_seconds"] = selected_detail[
                "absolute_start_difference_seconds"
            ]
        else:
            match_row["s3_bucket"] = ""
            match_row["s3_key"] = ""
            match_row["clip_timestamp_utc"] = ""
            match_row["clip_duration_seconds"] = np.nan
            match_row["clip_end_utc"] = ""
            match_row["event_offset_seconds"] = np.nan
            match_row["event_to_clip_seconds"] = np.nan

        matches.append(match_row)

    return pd.DataFrame(matches)


def build_event_window_clip_map(
    events: pd.DataFrame,
    videos: list[S3Video],
    duration_by_key: dict[str, float],
    event_window_seconds: float,
) -> pd.DataFrame:
    """Return one row per event and S3 clip overlapping its event window."""

    video_timestamps = [video.timestamp_utc for video in videos]
    rows = []

    for _, event in events.iterrows():
        event_time_utc = event["event_time_utc"].to_pydatetime().replace(tzinfo=None)
        window_start = event_time_utc - timedelta(seconds=event_window_seconds)
        window_end = event_time_utc + timedelta(seconds=event_window_seconds)
        candidates = candidate_videos_for_event_window(
            event_time_utc,
            videos,
            video_timestamps,
            event_window_seconds,
        )

        for video in candidates:
            duration_seconds = duration_by_key.get(video.key)
            clip_end = (
                video.timestamp_utc + timedelta(seconds=duration_seconds)
                if duration_seconds is not None
                else None
            )

            # A known interval must overlap the event window. If duration
            # probing failed, retain starts that are explicitly inside it.
            if clip_end is not None:
                overlaps_window = (
                    video.timestamp_utc <= window_end
                    and clip_end >= window_start
                )
                inclusion_method = "clip_interval_overlaps_event_window"
            else:
                overlaps_window = window_start <= video.timestamp_utc <= window_end
                inclusion_method = "clip_start_inside_window_duration_unknown"

            if not overlaps_window:
                continue

            row = event.to_dict()
            row.update({
                "s3_bucket": video.bucket,
                "s3_key": video.key,
                "clip_timestamp_utc": video.timestamp_utc.isoformat(),
                "clip_duration_seconds": duration_seconds,
                "clip_end_utc": clip_end.isoformat() if clip_end is not None else "",
                "event_window_start_utc": window_start.isoformat(),
                "event_window_end_utc": window_end.isoformat(),
                "event_to_clip_seconds": abs(
                    (video.timestamp_utc - event_time_utc).total_seconds()
                ),
                "event_offset_from_clip_start_seconds": (
                    event_time_utc - video.timestamp_utc
                ).total_seconds(),
                "window_inclusion_method": inclusion_method,
            })
            rows.append(row)

    return pd.DataFrame(rows)


---

## 3. Build the event-window S3 clip allowlist

For each stoppage, Chapter 3 considers all filename timestamps from one minute before to one minute after the PLC event. It also includes the immediately preceding clip because its duration may overlap the start of the window.

Candidate durations are remotely probed and cached. A clip is allowlisted when its interval overlaps the event window:

`clip_start <= event_window_end` and `clip_start + duration >= event_window_start`

The two absolute-closest filename timestamps are still audited to identify the anchor clip containing the exact PLC timestamp. Only the resulting event-window allowlist can enter the later download loop.

In [ ]:
events_csv = CFG.events_csv or find_latest_event_extract()
events_csv = Path(events_csv).resolve()
events = load_events(events_csv, CFG.event_timezone)
event_window_seconds = CFG.event_window_minutes * 60.0

# Apply the optional user-configured S3 timestamp bounds.
listing_start = S3_START_TIMESTAMP_UTC
listing_end = S3_END_TIMESTAMP_UTC

print(f"Events extract: {events_csv}")
print(f"Included events: {len(events):,}")
print(f"Event search window: {CFG.event_window_minutes:g} minute(s) each side")
print(f"S3 timestamp window: {s3_timestamp_window_text}")

# Listing keys does not save video content locally.
s3_client = create_s3_client(CFG.aws_profile)
s3_videos = list_s3_videos(
    s3_client,
    CFG.s3_bucket,
    CFG.s3_prefix,
    listing_start,
    listing_end,
)

# Probe durations for clips that could participate in an event window.
duration_candidates = collect_duration_candidates(
    events,
    s3_videos,
    event_window_seconds,
)
duration_cache_path = CFG.output_dir / "s3_clip_duration_cache.csv"
duration_by_key, duration_cache = load_or_probe_s3_durations(
    s3_client=s3_client,
    candidate_videos=duration_candidates,
    cache_path=duration_cache_path,
    timeout_seconds=CFG.duration_probe_timeout_seconds,
    max_workers=CFG.duration_probe_workers,
)

# Keep the exact-event anchor audit based on the two closest timestamps.
event_clip_matches = match_events_to_videos(
    events=events,
    videos=s3_videos,
    duration_by_key=duration_by_key,
    fallback_tolerance_seconds=CFG.match_tolerance_seconds,
)
event_clip_matches.to_csv(
    CFG.output_dir / "event_clip_matches.csv",
    index=False,
)
event_clip_matches.loc[~event_clip_matches["matched"]].to_csv(
    CFG.output_dir / "unmatched_events.csv",
    index=False,
)

# Build the many-to-many event-window map used for downloads and scoring.
event_window_clips = build_event_window_clip_map(
    events=events,
    videos=s3_videos,
    duration_by_key=duration_by_key,
    event_window_seconds=event_window_seconds,
)
event_window_clips_path = CFG.output_dir / "event_window_clips.csv"
event_window_clips.to_csv(event_window_clips_path, index=False)

if event_window_clips.empty:
    download_allowlist = pd.DataFrame(columns=["s3_bucket", "s3_key"])
else:
    download_allowlist = (
        event_window_clips.groupby(
            [
                "s3_bucket",
                "s3_key",
                "clip_timestamp_utc",
                "clip_duration_seconds",
                "clip_end_utc",
            ],
            as_index=False,
            dropna=False,
        )
        .agg(
            matched_event_count=("event_id", "size"),
            minimum_event_to_clip_seconds=("event_to_clip_seconds", "min"),
            event_ids=("event_id", lambda values: ";".join(values.astype(str))),
        )
        .sort_values("clip_timestamp_utc")
        .reset_index(drop=True)
    )

download_allowlist_path = CFG.output_dir / "s3_download_allowlist.csv"
download_allowlist.to_csv(download_allowlist_path, index=False)
ALLOWED_S3_KEYS = frozenset(download_allowlist["s3_key"].astype(str))

print(f"Timestamped videos listed:  {len(s3_videos):,}")
print(f"Duration candidates:        {len(duration_candidates):,}")
print(f"Durations resolved:         {len(duration_by_key):,}")
print(f"Event-window associations:  {len(event_window_clips):,}")
print(f"Unique permitted videos:    {len(ALLOWED_S3_KEYS):,}")
print(f"Duration cache:             {duration_cache_path}")
print(f"Event-window map:           {event_window_clips_path}")
print(f"Download allowlist:         {download_allowlist_path}")

display(download_allowlist.head(20))

---

## 4. CNN and temporal-window inference helpers

In [ ]:
class FrameBatchDataset(Dataset):
    """Apply the CNN preprocessing transform to one decoded frame batch."""

    def __init__(self, rgb_frames: list[np.ndarray], transform) -> None:
        self.rgb_frames = rgb_frames
        self.transform = transform

    def __len__(self) -> int:
        return len(self.rgb_frames)

    def __getitem__(self, index: int) -> torch.Tensor:
        image = Image.fromarray(self.rgb_frames[index])
        return self.transform(image)


def calculate_file_sha256(file_path: Path) -> str:
    """Calculate a checksum without loading the complete model into memory."""

    digest = hashlib.sha256()
    with Path(file_path).open("rb") as input_file:
        for chunk in iter(lambda: input_file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def parse_naive_utc_timestamp(value) -> datetime:
    """Parse a metadata UTC timestamp into the notebook's naive UTC convention."""

    timestamp = pd.Timestamp(value)
    if timestamp.tzinfo is not None:
        timestamp = timestamp.tz_convert("UTC").tz_localize(None)
    return timestamp.to_pydatetime()


def validate_model_handoff(metadata, model_path, config) -> None:
    """Refuse stale class mappings, orientation rules, or model weights."""

    expected_classes = ["no_bottle", "normal", "fallen_before_entry", "fallen_in_view"]
    actual_classes = list(metadata.get("class_names", []))
    if actual_classes != expected_classes:
        raise ValueError(
            f"Model classes must be {expected_classes}, found {actual_classes}"
        )

    if metadata.get("handoff_schema_version") != MODEL_HANDOFF_SCHEMA_VERSION:
        raise ValueError(
            "Model metadata uses the old stretched-image preprocessing. "
            "Retrain the model with aspect-preserving letterboxing."
        )

    metadata_preprocessing = metadata.get("image_preprocessing_version")
    if metadata_preprocessing != IMAGE_PREPROCESSING_VERSION:
        raise ValueError(
            "Training and inference image preprocessing do not match: "
            f"{metadata_preprocessing} != {IMAGE_PREPROCESSING_VERSION}"
        )

    metadata_cutoff = parse_naive_utc_timestamp(
        metadata["rotate_from_timestamp_utc"]
    )
    if metadata_cutoff != config.rotate_from_timestamp_utc:
        raise ValueError(
            "Training metadata and S3 rotation cutoffs do not match: "
            f"{metadata_cutoff} != {config.rotate_from_timestamp_utc}"
        )

    metadata_window_seconds = float(metadata["rolling_window_seconds"])
    if metadata_window_seconds != config.rolling_window_seconds:
        raise ValueError(
            "Training metadata and inference rolling windows do not match: "
            f"{metadata_window_seconds} != {config.rolling_window_seconds}"
        )

    metadata_sample_fps = float(metadata["inference_sample_fps"])
    if metadata_sample_fps != config.sample_fps:
        raise ValueError(
            "Training metadata and inference sampling targets do not match: "
            f"{metadata_sample_fps} != {config.sample_fps}"
        )

    metadata_aggregation = str(metadata["inference_aggregation_version"])
    if metadata_aggregation != TEMPORAL_AGGREGATION_VERSION:
        raise ValueError(
            "Training metadata and inference aggregation versions do not match: "
            f"{metadata_aggregation} != {TEMPORAL_AGGREGATION_VERSION}"
        )

    temporal_parameter_names = (
        "persistent_seconds",
        "min_falling_seconds",
        "peak_support_seconds",
        "normal_clear_seconds",
        "fallen_clear_seconds",
        "normal_threshold",
        "fault_activity_threshold",
        "falling_threshold",
        "falling_peak_threshold",
        "fallen_threshold",
    )
    for parameter_name in temporal_parameter_names:
        metadata_value = float(metadata[parameter_name])
        config_value = float(getattr(config, parameter_name))
        if not np.isclose(metadata_value, config_value, rtol=0.0, atol=1e-12):
            raise ValueError(
                f"Training and inference {parameter_name} values do not match: "
                f"{metadata_value} != {config_value}"
            )

    manifest_path = Path(metadata["training_frame_manifest"])
    expected_manifest_checksum = str(metadata.get("training_manifest_sha256", ""))
    actual_manifest_checksum = calculate_file_sha256(manifest_path)
    if actual_manifest_checksum != expected_manifest_checksum:
        raise ValueError(
            "The training manifest changed after the model was trained. "
            "Retrain all four classes before inference."
        )

    expected_checksum = str(metadata.get("model_checkpoint_sha256", ""))
    if not expected_checksum:
        raise ValueError("Model metadata does not contain a checkpoint checksum")
    actual_checksum = calculate_file_sha256(model_path)
    if actual_checksum != expected_checksum:
        raise ValueError(
            "best_model.pth does not match model_metadata.json. "
            "Rerun the final training metadata cell."
        )


def get_class_names(metadata: dict) -> list[str]:
    """Read class names from the current or an older metadata layout."""

    class_names = metadata.get("class_names") or metadata.get("classes")
    if not class_names:
        raise ValueError("Model metadata does not contain class names.")

    return list(class_names)


def build_model(metadata: dict, model_path: Path, device: torch.device) -> nn.Module:
    """Recreate the notebook's ResNet50 head and load its trained weights."""

    class_names = get_class_names(metadata)

    # Do not request pretrained weights because the checkpoint contains all weights.
    model = resnet50(weights=None)
    input_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.5),
        nn.Linear(input_features, 512),
        nn.ReLU(inplace=True),
        nn.Dropout(0.3),
        nn.Linear(512, len(class_names)),
    )

    try:
        state_dict = torch.load(
            model_path,
            map_location=device,
            weights_only=True,
        )
    except TypeError:
        # Support older PyTorch versions without the weights_only argument.
        state_dict = torch.load(model_path, map_location=device)

    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()

    return model


def build_inference_transform(metadata: dict):
    """Create the same aspect-preserving transform used during training."""

    image_size = int(metadata["img_size"])
    mean = metadata["imagenet_mean"]
    standard_deviation = metadata["imagenet_std"]
    fill_color = tuple(int(value) for value in metadata["letterbox_fill_rgb"])

    def letterbox_image(image):
        """Resize one image without distortion and pad it to a square."""
        image = image.convert("RGB")
        resized = ImageOps.contain(
            image,
            (image_size, image_size),
            method=Image.Resampling.BILINEAR,
        )
        canvas = Image.new("RGB", (image_size, image_size), fill_color)
        left = (image_size - resized.width) // 2
        top = (image_size - resized.height) // 2
        canvas.paste(resized, (left, top))
        return canvas

    return transforms.Compose(
        [
            letterbox_image,
            transforms.ToImage(),
            transforms.ToDtype(torch.float32, scale=True),
            transforms.Normalize(mean=mean, std=standard_deviation),
        ]
    )


def read_video_properties(video_path: Path) -> tuple[int, float]:
    """Read source frame count and FPS before inference."""

    capture = cv2.VideoCapture(str(video_path))
    if not capture.isOpened():
        capture.release()
        raise RuntimeError(f"OpenCV could not open {video_path.name}")

    try:
        frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
        fps = float(capture.get(cv2.CAP_PROP_FPS))
    finally:
        capture.release()

    if not np.isfinite(fps) or fps <= 0:
        fps = 30.0
    return max(0, frame_count), fps


def iter_frame_batches(
    video_path: Path,
    batch_size: int,
    rotate_180: bool,
    sample_fps: float | None,
) -> Iterator[tuple[list[int], float, list[np.ndarray]]]:
    """Yield every frame or a uniform FPS sample in bounded RGB batches."""

    capture = cv2.VideoCapture(str(video_path))
    if not capture.isOpened():
        capture.release()
        raise RuntimeError(f"OpenCV could not open {video_path.name}")

    fps = float(capture.get(cv2.CAP_PROP_FPS))
    if not np.isfinite(fps) or fps <= 0:
        fps = 30.0

    if sample_fps is None or sample_fps >= fps:
        frame_step = 1
    else:
        frame_step = max(1, int(round(fps / sample_fps)))

    frame_index = 0
    batch_frame_indices: list[int] = []
    rgb_frames: list[np.ndarray] = []

    try:
        while True:
            success, bgr_frame = capture.read()
            if not success:
                break

            should_classify_frame = frame_index % frame_step == 0
            if should_classify_frame:
                if rotate_180:
                    bgr_frame = cv2.rotate(bgr_frame, cv2.ROTATE_180)

                rgb_frame = cv2.cvtColor(bgr_frame, cv2.COLOR_BGR2RGB)
                batch_frame_indices.append(frame_index)
                rgb_frames.append(rgb_frame)

                if len(rgb_frames) == batch_size:
                    yield batch_frame_indices, fps, rgb_frames
                    batch_frame_indices = []
                    rgb_frames = []

            frame_index += 1

        if rgb_frames:
            yield batch_frame_indices, fps, rgb_frames
    finally:
        capture.release()


def infer_frame_batch(
    rgb_frames: list[np.ndarray],
    transform,
    model: nn.Module,
    device: torch.device,
    batch_size: int,
) -> np.ndarray:
    """Return class probabilities for every frame in one decoded batch."""

    dataset = FrameBatchDataset(rgb_frames, transform)
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=device.type == "cuda",
    )
    probability_batches: list[np.ndarray] = []

    with torch.inference_mode():
        for images in loader:
            images = images.to(device, non_blocking=device.type == "cuda")

            if device.type == "cuda":
                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    logits = model(images)
            else:
                logits = model(images)

            # Calculate softmax in float32 to avoid float16 saturation at 1.0.
            probabilities = torch.softmax(logits.float(), dim=1)
            probability_batches.append(probabilities.cpu().numpy())

    return np.concatenate(probability_batches, axis=0)



def _maximum_consecutive_frames(boolean_values):
    """Return the longest consecutive True run in one temporal window."""

    longest_run = 0
    current_run = 0
    for is_active in boolean_values:
        if is_active:
            current_run += 1
            longest_run = max(longest_run, current_run)
        else:
            current_run = 0
    return longest_run


def _frames_for_seconds(seconds, effective_sample_fps, minimum_frames=1):
    """Convert a duration to frames without floating-point boundary inflation."""

    frame_count = int(np.ceil(seconds * effective_sample_fps - 1e-9))
    return max(minimum_frames, frame_count)


def calculate_temporal_clip_decision(
    frame_probabilities: np.ndarray,
    frame_indices: list[int],
    source_fps: float,
    class_names: list[str],
    rolling_window_seconds: float,
    fallen_threshold: float,
    falling_threshold: float,
    normal_threshold: float = 0.75,
    fault_activity_threshold: float = 0.50,
    falling_peak_threshold: float = 0.80,
    persistent_seconds: float = 0.3,
    min_falling_seconds: float = 0.10,
    peak_support_seconds: float = 0.10,
    normal_clear_seconds: float = 0.5,
    fallen_clear_seconds: float = 0.75,
) -> dict:
    """Apply class-specific FPS-aware temporal rules to one classified clip."""

    probabilities = np.asarray(frame_probabilities, dtype=np.float64)
    indices = np.asarray(frame_indices, dtype=np.int64)
    if len(probabilities) == 0 or len(probabilities) != len(indices):
        raise ValueError("Frame probabilities and indices must be non-empty and aligned")
    if source_fps <= 0:
        raise ValueError("source_fps must be positive")

    no_bottle_index = class_names.index("no_bottle")
    normal_index = class_names.index("normal")
    fallen_index = class_names.index("fallen_before_entry")
    falling_index = class_names.index("fallen_in_view")
    frame_times = indices / float(source_fps)

    # Use the actual classified-frame rate, including any inference sampling.
    if len(indices) == 1:
        effective_sample_fps = float(source_fps)
    else:
        positive_index_steps = np.diff(indices)
        positive_index_steps = positive_index_steps[positive_index_steps > 0]
        effective_sample_fps = (
            float(source_fps / np.median(positive_index_steps))
            if len(positive_index_steps)
            else float(source_fps)
        )

    window_size = _frames_for_seconds(rolling_window_seconds, effective_sample_fps)
    persistent_size = _frames_for_seconds(persistent_seconds, effective_sample_fps)
    min_falling_frames = _frames_for_seconds(min_falling_seconds, effective_sample_fps, minimum_frames=2)
    support_radius_frames = _frames_for_seconds(peak_support_seconds, effective_sample_fps)
    normal_clear_frames = _frames_for_seconds(normal_clear_seconds, effective_sample_fps, minimum_frames=2)
    fallen_clear_frames = _frames_for_seconds(fallen_clear_seconds, effective_sample_fps, minimum_frames=2)
    window_size = min(window_size, len(probabilities))
    persistent_size = min(persistent_size, window_size)

    decision_windows = []
    current_state = "normal"
    stable_normal_frames = 0
    observed_falling_in_view = False

    for window_end in range(window_size - 1, len(probabilities)):
        window_start = window_end - window_size + 1
        history = probabilities[window_start : window_end + 1]
        persistent_history = history[-persistent_size:]

        no_bottle_values = history[:, no_bottle_index]
        normal_values = history[:, normal_index]
        fallen_values = persistent_history[:, fallen_index]
        falling_values = history[:, falling_index]

        no_bottle_mean = float(np.mean(no_bottle_values))
        normal_mean = float(np.mean(normal_values))
        safe_classification = (
            "no_bottle" if no_bottle_mean >= normal_mean else "normal"
        )
        safe_mean = max(no_bottle_mean, normal_mean)
        safe_values = (
            no_bottle_values if safe_classification == "no_bottle" else normal_values
        )
        fallen_median = float(np.median(fallen_values))
        falling_peak_relative = int(np.argmax(falling_values))
        falling_peak = float(falling_values[falling_peak_relative])
        falling_peak_position = window_start + falling_peak_relative

        falling_active = falling_values >= falling_threshold
        falling_consecutive_frames = _maximum_consecutive_frames(falling_active)

        local_start = max(0, falling_peak_relative - support_radius_frames)
        local_stop = min(len(falling_values), falling_peak_relative + support_radius_frames + 1)
        falling_support_frames = int(np.count_nonzero(
            falling_values[local_start:local_stop] >= fault_activity_threshold
        ))
        falling_frames_above_activity = int(np.count_nonzero(
            falling_values >= fault_activity_threshold
        ))
        fallen_frames_above_activity = int(np.count_nonzero(
            history[:, fallen_index] >= fault_activity_threshold
        ))

        # Fault evidence takes precedence over normal evidence.
        if fallen_median >= fallen_threshold:
            candidate_classification = "fallen_before_entry"
            candidate_score = fallen_median
            persistent_start = window_end - persistent_size + 1
            selected_relative = int(np.argmax(
                probabilities[persistent_start : window_end + 1, fallen_index]
            ))
            selected_frame_position = persistent_start + selected_relative
        elif falling_consecutive_frames >= min_falling_frames:
            candidate_classification = "fallen_in_view"
            candidate_score = falling_peak
            selected_frame_position = falling_peak_position
        elif (
            falling_peak >= falling_peak_threshold
            and falling_support_frames >= 2
        ):
            candidate_classification = "fallen_in_view"
            candidate_score = falling_peak
            selected_frame_position = falling_peak_position
        elif (
            safe_mean >= normal_threshold
            and falling_frames_above_activity < 2
            and fallen_frames_above_activity < 2
        ):
            candidate_classification = safe_classification
            candidate_score = safe_mean
            selected_frame_position = window_start + int(np.argmax(safe_values))
        else:
            candidate_classification = "uncertain"
            candidate_score = float(np.max(history[-1]))
            selected_frame_position = window_end

        # Preserve how the fault began, and require stable normal evidence to clear it.
        if candidate_classification == "fallen_in_view":
            observed_falling_in_view = True
            current_state = "fallen_in_view"
            stable_normal_frames = 0
        elif candidate_classification == "fallen_before_entry":
            if not observed_falling_in_view:
                current_state = "fallen_before_entry"
            stable_normal_frames = 0
        elif candidate_classification in {"normal", "no_bottle"}:
            stable_normal_frames += 1
            required_clear_frames = (
                fallen_clear_frames
                if current_state == "fallen_before_entry"
                else normal_clear_frames
            )
            if (
                current_state in {"normal", "no_bottle", "uncertain"}
                or stable_normal_frames >= required_clear_frames
            ):
                current_state = candidate_classification
        else:
            stable_normal_frames = 0
            if current_state in {"normal", "no_bottle"}:
                current_state = "uncertain"

        decision_windows.append({
            "candidate_classification": candidate_classification,
            "state": current_state,
            "score": candidate_score,
            "window_start_position": window_start,
            "window_end_position": window_end,
            "selected_frame_position": selected_frame_position,
            "no_bottle_mean": no_bottle_mean,
            "normal_mean": normal_mean,
            "fallen_median": fallen_median,
            "falling_peak": falling_peak,
            "falling_consecutive_frames": falling_consecutive_frames,
            "falling_support_frames": falling_support_frames,
        })

    falling_windows = [
        item for item in decision_windows
        if item["candidate_classification"] == "fallen_in_view"
    ]
    fallen_windows = [
        item for item in decision_windows
        if item["candidate_classification"] == "fallen_before_entry"
    ]
    safe_windows = [
        item for item in decision_windows
        if item["candidate_classification"] in {"normal", "no_bottle"}
    ]

    # A detected in-view fall remains the clip's event origin even after it settles.
    if falling_windows:
        prediction = "fallen_in_view"
        selected_window = max(falling_windows, key=lambda item: item["score"])
    elif fallen_windows:
        prediction = "fallen_before_entry"
        selected_window = max(fallen_windows, key=lambda item: item["score"])
    elif safe_windows:
        selected_window = max(safe_windows, key=lambda item: item["score"])
        prediction = selected_window["candidate_classification"]
    else:
        prediction = "uncertain"
        selected_window = max(decision_windows, key=lambda item: item["score"])

    peak_frame_position = int(selected_window["selected_frame_position"])
    peak_window_position = int(selected_window["window_start_position"])
    peak_window_end_position = int(selected_window["window_end_position"])
    confidence = float(selected_window["score"])

    strongest_fallen_window = max(decision_windows, key=lambda item: item["fallen_median"])
    strongest_falling_window = max(decision_windows, key=lambda item: item["falling_peak"])
    temporal_scores = {
        "no_bottle": max(item["no_bottle_mean"] for item in decision_windows),
        "normal": max(item["normal_mean"] for item in decision_windows),
        "fallen_before_entry": strongest_fallen_window["fallen_median"],
        "fallen_in_view": strongest_falling_window["falling_peak"],
    }

    detection_events = [
        item for item in decision_windows
        if item["candidate_classification"] in {"fallen_before_entry", "fallen_in_view"}
    ]

    return {
        "prediction": prediction,
        "confidence": confidence,
        "temporal_scores": temporal_scores,
        "peak_anomaly_score": confidence,
        "peak_anomaly_class": prediction,
        "peak_frame_anomaly_probability": float(
            probabilities[peak_frame_position].max()
        ),
        "peak_anomaly_frame_index": int(indices[peak_frame_position]),
        "peak_anomaly_frame_time_seconds": float(frame_times[peak_frame_position]),
        "effective_sample_fps": effective_sample_fps,
        "rolling_window_frame_count": int(window_size),
        "persistent_frame_count": int(persistent_size),
        "min_falling_frame_count": int(min_falling_frames),
        "peak_window_start_frame": int(indices[peak_window_position]),
        "peak_window_end_frame": int(indices[peak_window_end_position]),
        "peak_window_start_seconds": float(frame_times[peak_window_position]),
        "peak_window_end_seconds": float(frame_times[peak_window_end_position]),
        "strongest_entry_window_start_frame": int(indices[strongest_fallen_window["window_start_position"]]),
        "strongest_entry_window_end_frame": int(indices[strongest_fallen_window["window_end_position"]]),
        "strongest_entry_window_start_seconds": float(frame_times[strongest_fallen_window["window_start_position"]]),
        "strongest_entry_window_end_seconds": float(frame_times[strongest_fallen_window["window_end_position"]]),
        "strongest_fall_window_start_frame": int(indices[strongest_falling_window["window_start_position"]]),
        "strongest_fall_window_end_frame": int(indices[strongest_falling_window["window_end_position"]]),
        "strongest_fall_window_start_seconds": float(frame_times[strongest_falling_window["window_start_position"]]),
        "strongest_fall_window_end_seconds": float(frame_times[strongest_falling_window["window_end_position"]]),
        "decision_windows": decision_windows,
        "detection_events": detection_events,
        "preview_position": peak_frame_position,
        "strongest_entry_window": {
            "start_frame": int(indices[strongest_fallen_window["window_start_position"]]),
            "end_frame": int(indices[strongest_fallen_window["window_end_position"]]),
            "start_seconds": float(frame_times[strongest_fallen_window["window_start_position"]]),
            "end_seconds": float(frame_times[strongest_fallen_window["window_end_position"]]),
        },
        "strongest_fall_window": {
            "start_frame": int(indices[strongest_falling_window["window_start_position"]]),
            "end_frame": int(indices[strongest_falling_window["window_end_position"]]),
            "start_seconds": float(frame_times[strongest_falling_window["window_start_position"]]),
            "end_seconds": float(frame_times[strongest_falling_window["window_end_position"]]),
        },
    }

def should_rotate_video_180(video_timestamp_utc, rotate_from_timestamp_utc):
    """Rotate videos at or after the configured UTC camera cutoff."""

    return video_timestamp_utc >= rotate_from_timestamp_utc


def classify_all_frames(
    video: S3Video,
    video_path: Path,
    model: nn.Module,
    transform,
    metadata: dict,
    device: torch.device,
    config,
) -> tuple[list[dict], dict]:
    """Classify frames and apply class-specific temporal aggregation."""

    class_names = get_class_names(metadata)

    # Apply the orientation correction once, before any frame preprocessing.
    rotate_180 = should_rotate_video_180(
        video.timestamp_utc,
        config.rotate_from_timestamp_utc,
    )

    source_frame_count, video_fps = read_video_properties(video_path)
    frame_rows: list[dict] = []
    classified_frame_indices: list[int] = []
    probability_batches: list[np.ndarray] = []

    for batch_indices, video_fps, rgb_frames in iter_frame_batches(
        video_path,
        config.batch_size,
        rotate_180,
        config.sample_fps,
    ):
        probabilities = infer_frame_batch(
            rgb_frames,
            transform,
            model,
            device,
            config.batch_size,
        )
        classified_frame_indices.extend(batch_indices)
        probability_batches.append(probabilities)

        for frame_index, frame_probabilities in zip(batch_indices, probabilities):
            predicted_index = int(np.argmax(frame_probabilities))
            row = {
                "s3_bucket": video.bucket,
                "s3_key": video.key,
                "clip_timestamp_utc": video.timestamp_utc.isoformat(),
                "frame_index": frame_index,
                "frame_time_seconds": frame_index / video_fps,
                "predicted_class": class_names[predicted_index],
                "confidence": float(frame_probabilities[predicted_index]),
            }

            for class_index, class_name in enumerate(class_names):
                row[f"probability_{class_name}"] = float(
                    frame_probabilities[class_index]
                )

            frame_rows.append(row)

    if not probability_batches:
        raise RuntimeError(f"No frames were decoded from {video_path.name}")

    frame_probabilities = np.concatenate(probability_batches, axis=0)
    temporal_result = calculate_temporal_clip_decision(
        frame_probabilities=frame_probabilities,
        frame_indices=classified_frame_indices,
        source_fps=video_fps,
        class_names=class_names,
        rolling_window_seconds=config.rolling_window_seconds,
        fallen_threshold=config.fallen_threshold,
        falling_threshold=config.falling_threshold,
        normal_threshold=config.normal_threshold,
        fault_activity_threshold=config.fault_activity_threshold,
        falling_peak_threshold=config.falling_peak_threshold,
        persistent_seconds=config.persistent_seconds,
        min_falling_seconds=config.min_falling_seconds,
        peak_support_seconds=config.peak_support_seconds,
        normal_clear_seconds=config.normal_clear_seconds,
        fallen_clear_seconds=config.fallen_clear_seconds,
    )

    # Read all three probabilities at the exact frame selected for the image.
    peak_frame_index = temporal_result["peak_anomaly_frame_index"]
    peak_frame_position = classified_frame_indices.index(peak_frame_index)
    peak_frame_probabilities = frame_probabilities[peak_frame_position]
    peak_frame_scores = {
        class_name: float(peak_frame_probabilities[class_index])
        for class_index, class_name in enumerate(class_names)
    }

    # Some containers do not expose frame count metadata, so use decoded evidence.
    if source_frame_count <= 0:
        source_frame_count = classified_frame_indices[-1] + 1

    clip_result = {
        "s3_bucket": video.bucket,
        "s3_key": video.key,
        "clip_timestamp_utc": video.timestamp_utc.isoformat(),
        "status": "success",
        "error": "",
        "decoded_frame_count": source_frame_count,
        "classified_frame_count": len(classified_frame_indices),
        "fps": video_fps,
        "effective_sample_fps": temporal_result["effective_sample_fps"],
        "duration_seconds": source_frame_count / video_fps,
        "rotated_180": rotate_180,
        "aggregation_version": TEMPORAL_AGGREGATION_VERSION,
        "rolling_window_seconds": config.rolling_window_seconds,
        "fallen_threshold": config.fallen_threshold,
        "falling_threshold": config.falling_threshold,
        "normal_threshold": config.normal_threshold,
        "fault_activity_threshold": config.fault_activity_threshold,
        "persistent_seconds": config.persistent_seconds,
        "min_falling_seconds": config.min_falling_seconds,
        "peak_anomaly_score": temporal_result["peak_anomaly_score"],
        "peak_anomaly_class": temporal_result["peak_anomaly_class"],
        "peak_frame_anomaly_probability": temporal_result["peak_frame_anomaly_probability"],
        "peak_frame_score_no_bottle": peak_frame_scores["no_bottle"],
        "peak_frame_score_normal": peak_frame_scores["normal"],
        "peak_frame_score_fallen_before_entry": peak_frame_scores[
            "fallen_before_entry"
        ],
        "peak_frame_score_fallen_in_view": peak_frame_scores["fallen_in_view"],
        "peak_anomaly_frame_index": temporal_result["peak_anomaly_frame_index"],
        "peak_anomaly_frame_time_seconds": temporal_result["peak_anomaly_frame_time_seconds"],
        "peak_window_start_frame": temporal_result["peak_window_start_frame"],
        "peak_window_end_frame": temporal_result["peak_window_end_frame"],
        "peak_window_start_seconds": temporal_result["peak_window_start_seconds"],
        "peak_window_end_seconds": temporal_result["peak_window_end_seconds"],
        "strongest_entry_window_start_frame": temporal_result["strongest_entry_window_start_frame"],
        "strongest_entry_window_end_frame": temporal_result["strongest_entry_window_end_frame"],
        "strongest_entry_window_start_seconds": temporal_result["strongest_entry_window_start_seconds"],
        "strongest_entry_window_end_seconds": temporal_result["strongest_entry_window_end_seconds"],
        "strongest_fall_window_start_frame": temporal_result["strongest_fall_window_start_frame"],
        "strongest_fall_window_end_frame": temporal_result["strongest_fall_window_end_frame"],
        "strongest_fall_window_start_seconds": temporal_result["strongest_fall_window_start_seconds"],
        "strongest_fall_window_end_seconds": temporal_result["strongest_fall_window_end_seconds"],
        "predicted_class": temporal_result["prediction"],
        "confidence": temporal_result["confidence"],
    }

    for class_name in class_names:
        clip_result[f"temporal_score_{class_name}"] = temporal_result[
            "temporal_scores"
        ][class_name]

    return frame_rows, clip_result


---

## 5. Resumable download and CSV helpers

Every successfully classified video is retained locally under:

- `classified_videos/no_bottle/`
- `classified_videos/normal/`
- `classified_videos/fallen_before_entry/`
- `classified_videos/fallen_in_view/`
- `classified_videos/uncertain/`

Downloads remain restricted to the Chapter 3 allowlist.

In [ ]:
def append_rows(csv_path: Path, rows: list[dict]) -> None:
    """Append dictionaries to a CSV while preserving one stable header."""

    if not rows:
        return

    csv_path.parent.mkdir(parents=True, exist_ok=True)
    file_exists = csv_path.exists() and csv_path.stat().st_size > 0
    fieldnames = list(rows[0].keys())

    if file_exists:
        with csv_path.open("r", newline="", encoding="utf-8-sig") as input_file:
            existing_header = next(csv.reader(input_file), [])

        if existing_header != fieldnames:
            raise ValueError(
                f"The existing CSV header does not match the current output: {csv_path}"
            )

    with csv_path.open("a", newline="", encoding="utf-8-sig") as output_file:
        writer = csv.DictWriter(output_file, fieldnames=fieldnames)
        if not file_exists:
            writer.writeheader()
        writer.writerows(rows)


def load_successful_s3_rotations(
    clip_results_path: Path,
) -> dict[str, set[bool]]:
    """Read the successful rotation states already completed for each S3 key."""

    if not clip_results_path.exists():
        return {}

    results = pd.read_csv(clip_results_path)
    required_columns = {"s3_key", "status", "rotated_180"}
    if not required_columns.issubset(results.columns):
        return {}

    successful_results = results.loc[
        results["status"].astype(str).str.casefold().eq("success")
    ].copy()
    successful_results["rotation_value"] = successful_results[
        "rotated_180"
    ].map(
        lambda value: str(value).strip().casefold() in {"true", "1", "yes"}
    )

    rotations_by_key: dict[str, set[bool]] = {}
    for s3_key, key_rows in successful_results.groupby("s3_key"):
        rotations_by_key[str(s3_key)] = set(key_rows["rotation_value"])
    return rotations_by_key


def download_video(s3_client, video: S3Video, downloads_dir: Path) -> Path:
    """Download one S3 video atomically."""

    downloads_dir.mkdir(parents=True, exist_ok=True)
    key_hash = hashlib.sha1(video.key.encode("utf-8")).hexdigest()[:10]
    source_name = Path(video.key).name
    local_filename = (
        f"{Path(source_name).stem}_{key_hash}{Path(source_name).suffix}"
    )
    destination = downloads_dir / local_filename
    partial_destination = destination.with_suffix(destination.suffix + ".part")

    try:
        s3_client.download_file(
            video.bucket,
            video.key,
            str(partial_destination),
        )
        partial_destination.replace(destination)
    except (BotoCoreError, ClientError, TokenRetrievalError) as error:
        partial_destination.unlink(missing_ok=True)
        raise RuntimeError(f"Failed to download s3://{video.bucket}/{video.key}") from error

    return destination


def add_event_context(clip_result: dict, clip_events: pd.DataFrame) -> dict:
    """Add matching PLC event details to a clip result row."""

    result = clip_result.copy()
    result["matched_event_count"] = len(clip_events)
    result["event_ids"] = ";".join(clip_events["event_id"].astype(str))

    if "Order" in clip_events.columns:
        result["orders"] = ";".join(
            sorted(set(clip_events["Order"].dropna().astype(str)))
        )
    else:
        result["orders"] = ""

    result["minimum_event_to_clip_seconds"] = float(
        clip_events["event_to_clip_seconds"].min()
    )

    return result


def keep_or_remove_video(
    downloaded_path: Path,
    predicted_class: str,
    output_dir: Path,
) -> Path:
    """Move every classified video into its predicted-class folder."""

    destination_dir = output_dir / "classified_videos" / predicted_class
    destination_dir.mkdir(parents=True, exist_ok=True)
    destination = destination_dir / downloaded_path.name

    # A resumed run can safely reuse an already-retained identical S3 key.
    if destination.exists():
        downloaded_path.unlink(missing_ok=True)
        return destination

    shutil.move(str(downloaded_path), str(destination))
    return destination


def save_annotated_peak_anomaly_frame(
    video_path,
    frame_index,
    rotate_180,
    predicted_class,
    peak_anomaly_score,
    peak_frame_scores,
    estimated_timestamp_utc,
    output_dir,
):
    """Save the peak-anomaly source frame with its classification overlay."""

    video_path = Path(video_path)
    capture = cv2.VideoCapture(str(video_path))
    if not capture.isOpened():
        capture.release()
        raise RuntimeError(f"Could not open retained video: {video_path}")

    try:
        capture.set(cv2.CAP_PROP_POS_FRAMES, int(frame_index))
        success, frame = capture.read()
    finally:
        capture.release()

    if not success:
        raise RuntimeError(
            f"Could not extract peak frame {frame_index} from {video_path.name}"
        )
    should_rotate = (
        rotate_180 is True
        or str(rotate_180).strip().casefold() in {"true", "1", "yes"}
    )
    if should_rotate:
        frame = cv2.rotate(frame, cv2.ROTATE_180)

    timestamp_text = pd.Timestamp(estimated_timestamp_utc).isoformat()
    annotation_lines = [
        f"Classification: {predicted_class} ({peak_anomaly_score:.5f})",
        f"Frame no_bottle: {peak_frame_scores['no_bottle']:.5f}",
        f"Frame normal: {peak_frame_scores['normal']:.5f}",
        (
            "Frame fallen_before_entry: "
            f"{peak_frame_scores['fallen_before_entry']:.5f}"
        ),
        f"Frame fallen_in_view: {peak_frame_scores['fallen_in_view']:.5f}",
        f"UTC: {timestamp_text}",
    ]

    # Draw one compact outline around the annotation instead of covering the image.
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 0.65
    thickness = 2
    line_height = 28
    box_padding = 10
    text_widths = [
        cv2.getTextSize(annotation, font, font_scale, thickness)[0][0]
        for annotation in annotation_lines
    ]
    box_width = min(frame.shape[1] - 1, max(text_widths) + 2 * box_padding)
    box_height = min(
        frame.shape[0] - 1,
        box_padding + line_height * len(annotation_lines),
    )
    cv2.rectangle(frame, (0, 0), (box_width, box_height), (0, 0, 0), 2)
    for line_number, annotation in enumerate(annotation_lines):
        y_position = 24 + line_number * line_height

        # Add a thin black text outline so the label remains readable on the frame.
        cv2.putText(
            frame,
            annotation,
            (box_padding, y_position),
            font,
            font_scale,
            (0, 0, 0),
            thickness + 2,
            cv2.LINE_AA,
        )
        cv2.putText(
            frame,
            annotation,
            (box_padding, y_position),
            font,
            font_scale,
            (255, 255, 255),
            thickness,
            cv2.LINE_AA,
        )

    destination_dir = Path(output_dir) / "peak_anomaly_frames" / predicted_class
    destination_dir.mkdir(parents=True, exist_ok=True)
    destination_path = destination_dir / (
        f"{video_path.stem}_peak_frame_{int(frame_index):06d}.jpg"
    )
    if not cv2.imwrite(str(destination_path), frame):
        raise RuntimeError(f"Could not save peak anomaly frame: {destination_path}")
    return destination_path



In [ ]:
def classify_allowlisted_s3_clips(
    config,
    s3_client,
    s3_videos,
    event_window_clips,
    allowed_s3_keys,
):
    """Download and classify only event-matched S3 keys."""

    unique_videos = sorted(
        [video for video in s3_videos if video.key in allowed_s3_keys],
        key=lambda video: video.timestamp_utc,
    )
    if config.max_clips is not None:
        unique_videos = unique_videos[: config.max_clips]

    # This assertion is the hard download boundary.
    unexpected_keys = {
        video.key for video in unique_videos
        if video.key not in allowed_s3_keys
    }
    if unexpected_keys:
        raise RuntimeError(
            f"Refusing to download keys outside the event allowlist: {sorted(unexpected_keys)[:3]}"
        )

    if not unique_videos:
        print("No event-matched clips are available to classify.")
        return

    metadata = json.loads(config.metadata_path.read_text(encoding="utf-8"))
    validate_model_handoff(metadata, config.model_path, config)
    print("Model handoff validated: classes, rotation cutoff and checksum")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = build_model(metadata, config.model_path, device)
    transform = build_inference_transform(metadata)
    print(f"Classifier device: {device}")

    downloads_dir = config.output_dir / "downloads"
    frame_results_path = config.output_dir / "frame_classification_results.csv"
    clip_results_path = config.output_dir / "clip_classification_results.csv"
    if config.reprocess:
        # Start a clean result set instead of appending duplicate rows.
        frame_results_path.unlink(missing_ok=True)
        clip_results_path.unlink(missing_ok=True)
        successful_rotations_by_key = {}
    else:
        successful_rotations_by_key = load_successful_s3_rotations(
            clip_results_path
        )

    for clip_number, video in enumerate(unique_videos, start=1):
        # Recheck immediately before every possible download.
        if video.key not in allowed_s3_keys:
            raise RuntimeError(f"Blocked non-event S3 key: {video.key}")

        expected_rotate_180 = should_rotate_video_180(
            video.timestamp_utc,
            config.rotate_from_timestamp_utc,
        )
        completed_rotations = successful_rotations_by_key.get(video.key, set())
        if expected_rotate_180 in completed_rotations and not config.reprocess:
            print(f"[{clip_number}/{len(unique_videos)}] Skipping completed: {video.key}")
            continue

        print(f"[{clip_number}/{len(unique_videos)}] Processing: {video.key}")
        downloaded_path = None

        try:
            downloaded_path = download_video(s3_client, video, downloads_dir)
            frame_rows, clip_result = classify_all_frames(
                video=video,
                video_path=downloaded_path,
                model=model,
                transform=transform,
                metadata=metadata,
                device=device,
                config=config,
            )

            clip_events = event_window_clips.loc[
                event_window_clips["s3_key"] == video.key
            ]
            clip_result = add_event_context(clip_result, clip_events)

            saved_path = keep_or_remove_video(
                downloaded_path,
                clip_result["predicted_class"],
                config.output_dir,
            )
            downloaded_path = None
            clip_result["saved_video_path"] = str(saved_path)

            peak_timestamp_utc = video.timestamp_utc + timedelta(
                seconds=clip_result["peak_anomaly_frame_time_seconds"]
            )
            try:
                peak_frame_path = save_annotated_peak_anomaly_frame(
                    video_path=saved_path,
                    frame_index=clip_result["peak_anomaly_frame_index"],
                    rotate_180=clip_result["rotated_180"],
                    predicted_class=clip_result["predicted_class"],
                    peak_anomaly_score=clip_result["peak_anomaly_score"],
                    peak_frame_scores={
                        "no_bottle": clip_result["peak_frame_score_no_bottle"],
                        "normal": clip_result["peak_frame_score_normal"],
                        "fallen_before_entry": clip_result[
                            "peak_frame_score_fallen_before_entry"
                        ],
                        "fallen_in_view": clip_result[
                            "peak_frame_score_fallen_in_view"
                        ],
                    },
                    estimated_timestamp_utc=peak_timestamp_utc,
                    output_dir=config.output_dir,
                )
                clip_result["peak_anomaly_frame_path"] = str(peak_frame_path)
                clip_result["peak_anomaly_frame_error"] = ""
            except Exception as peak_frame_error:
                clip_result["peak_anomaly_frame_path"] = ""
                clip_result["peak_anomaly_frame_error"] = str(peak_frame_error)

            # Commit results only after the complete clip succeeds.
            append_rows(frame_results_path, frame_rows)
            append_rows(clip_results_path, [clip_result])
            successful_rotations_by_key.setdefault(video.key, set()).add(
                clip_result["rotated_180"]
            )

            print(
                f"  {clip_result['predicted_class']} "
                f"({clip_result['confidence']:.3f}), "
                f"{clip_result['classified_frame_count']}/"
                f"{clip_result['decoded_frame_count']} frames classified"
            )
        except Exception as error:
            if downloaded_path is not None:
                downloaded_path.unlink(missing_ok=True)

            failure_result = {
                "s3_bucket": video.bucket,
                "s3_key": video.key,
                "clip_timestamp_utc": video.timestamp_utc.isoformat(),
                "status": "failed",
                "error": str(error),
                "decoded_frame_count": "",
                "classified_frame_count": "",
                "fps": "",
                "effective_sample_fps": "",
                "duration_seconds": "",
                "rotated_180": expected_rotate_180,
                "aggregation_version": TEMPORAL_AGGREGATION_VERSION,
                "rolling_window_seconds": config.rolling_window_seconds,
                "fallen_threshold": config.fallen_threshold,
                "falling_threshold": config.falling_threshold,
                "normal_threshold": config.normal_threshold,
                "fault_activity_threshold": config.fault_activity_threshold,
                "persistent_seconds": config.persistent_seconds,
                "min_falling_seconds": config.min_falling_seconds,
                "peak_anomaly_score": "",
                "peak_anomaly_class": "",
                "peak_frame_anomaly_probability": "",
                "peak_frame_score_no_bottle": "",
                "peak_frame_score_normal": "",
                "peak_frame_score_fallen_before_entry": "",
                "peak_frame_score_fallen_in_view": "",
                "peak_anomaly_frame_index": "",
                "peak_anomaly_frame_time_seconds": "",
                "peak_window_start_frame": "",
                "peak_window_end_frame": "",
                "peak_window_start_seconds": "",
                "peak_window_end_seconds": "",
                "strongest_entry_window_start_frame": "",
                "strongest_entry_window_end_frame": "",
                "strongest_entry_window_start_seconds": "",
                "strongest_entry_window_end_seconds": "",
                "strongest_fall_window_start_frame": "",
                "strongest_fall_window_end_frame": "",
                "strongest_fall_window_start_seconds": "",
                "strongest_fall_window_end_seconds": "",
                "predicted_class": "",
                "confidence": "",
            }
            for class_name in get_class_names(metadata):
                failure_result[f"temporal_score_{class_name}"] = ""

            clip_events = event_window_clips.loc[
                event_window_clips["s3_key"] == video.key
            ]
            failure_result = add_event_context(failure_result, clip_events)
            failure_result["saved_video_path"] = ""
            failure_result["peak_anomaly_frame_path"] = ""
            failure_result["peak_anomaly_frame_error"] = ""
            append_rows(clip_results_path, [failure_result])
            print(f"  Failed: {error}")

    print(f"Classification complete. Results: {config.output_dir}")

---

## 6. Run the allowlisted S3 batch

Authenticate first if required:

```powershell
aws sso login --profile DashcamGlbDiageoProdDataContrib-522196013725
```

For an initial test, set `run_s3_batch=True` and `max_clips=5` in the configuration cell. Remove the limit after checking the outputs.

In [ ]:
if CFG.run_s3_batch:
    classify_allowlisted_s3_clips(
        config=CFG,
        s3_client=s3_client,
        s3_videos=s3_videos,
        event_window_clips=event_window_clips,
        allowed_s3_keys=ALLOWED_S3_KEYS,
    )
else:
    print("S3 classification is disabled. Set CFG.run_s3_batch=True in the configuration cell.")

---

## 7. Classify each stoppage from its first non-normal event

This stage combines frame results from every clip overlapping the event's Ã‚Â±N-minute window.

For each stoppage it:

1. Orders classified frames by estimated UTC timestamp.
2. Calculates complete rolling-window scores for both anomaly classes.
3. Finds the earliest window where either class reaches its own configured threshold.
4. Assigns the stoppage to that first non-normal class.
5. Extracts the first threshold-crossing frame.
6. Estimates the exact event timestamp as `S3 filename clip start + frame time`.

Extracted evidence frames are saved under `stoppage_event_frames/<class>/`.

In [ ]:
def find_first_non_normal_detection(event_frame_rows, config):
    """Return the earliest threshold-qualified fault using the shared temporal rules."""

    detection_candidates = []
    clip_predictions = []
    maximum_entry_score = 0.0
    maximum_fall_score = 0.0
    maximum_no_bottle_score = 0.0
    maximum_normal_score = 0.0
    class_names = ["no_bottle", "normal", "fallen_before_entry", "fallen_in_view"]

    for s3_key, clip_frames in event_frame_rows.groupby("s3_key", sort=False):
        clip_frames = clip_frames.sort_values("frame_index").reset_index(drop=True)
        if clip_frames.empty:
            continue

        frame_indices = clip_frames["frame_index"].to_numpy(dtype=np.int64)
        frame_times_seconds = clip_frames["frame_time_seconds"].to_numpy(dtype=float)
        probability_matrix = clip_frames[
            [
                "probability_no_bottle",
                "probability_normal",
                "probability_fallen_before_entry",
                "probability_fallen_in_view",
            ]
        ].to_numpy(dtype=np.float64)

        # Recover the source FPS from source-frame indices and saved frame times.
        if len(frame_indices) == 1:
            source_fps = 1.0
        else:
            time_steps = np.diff(frame_times_seconds)
            index_steps = np.diff(frame_indices)
            valid_steps = (time_steps > 0) & (index_steps > 0)
            source_fps = (
                float(np.median(index_steps[valid_steps] / time_steps[valid_steps]))
                if np.any(valid_steps)
                else 1.0
            )

        temporal_result = calculate_temporal_clip_decision(
            frame_probabilities=probability_matrix,
            frame_indices=frame_indices.tolist(),
            source_fps=source_fps,
            class_names=class_names,
            rolling_window_seconds=config.rolling_window_seconds,
            fallen_threshold=config.fallen_threshold,
            falling_threshold=config.falling_threshold,
            normal_threshold=config.normal_threshold,
            fault_activity_threshold=config.fault_activity_threshold,
            falling_peak_threshold=config.falling_peak_threshold,
            persistent_seconds=config.persistent_seconds,
            min_falling_seconds=config.min_falling_seconds,
            peak_support_seconds=config.peak_support_seconds,
            normal_clear_seconds=config.normal_clear_seconds,
            fallen_clear_seconds=config.fallen_clear_seconds,
        )
        clip_predictions.append(temporal_result["prediction"])
        maximum_no_bottle_score = max(
            maximum_no_bottle_score,
            float(temporal_result["temporal_scores"]["no_bottle"]),
        )
        maximum_normal_score = max(
            maximum_normal_score,
            float(temporal_result["temporal_scores"]["normal"]),
        )
        maximum_entry_score = max(
            maximum_entry_score,
            float(temporal_result["temporal_scores"]["fallen_before_entry"]),
        )
        maximum_fall_score = max(
            maximum_fall_score,
            float(temporal_result["temporal_scores"]["fallen_in_view"]),
        )

        for event in temporal_result["detection_events"]:
            detected_class = event["candidate_classification"]
            selected_position = int(event["selected_frame_position"])
            selected_frame = clip_frames.iloc[selected_position]
            class_probability_column = f"probability_{detected_class}"
            window_start_position = int(event["window_start_position"])
            window_end_position = int(event["window_end_position"])

            detection_candidates.append({
                "detected_class": detected_class,
                "detected_score": float(event["score"]),
                "selected_frame": selected_frame,
                "selected_frame_class_probability": float(
                    selected_frame[class_probability_column]
                ),
                "estimated_timestamp_utc": selected_frame[
                    "estimated_frame_timestamp_utc"
                ],
                "rolling_window_start_utc": clip_frames.iloc[
                    window_start_position
                ]["estimated_frame_timestamp_utc"],
                "rolling_window_end_utc": clip_frames.iloc[
                    window_end_position
                ]["estimated_frame_timestamp_utc"],
            })

    maximum_anomaly_score = max(maximum_entry_score, maximum_fall_score)
    if not detection_candidates:
        safe_predictions = {"normal", "no_bottle"}
        if clip_predictions and all(item in safe_predictions for item in clip_predictions):
            prediction = (
                "no_bottle"
                if all(item == "no_bottle" for item in clip_predictions)
                else "normal"
            )
        else:
            prediction = "uncertain"
        if prediction == "no_bottle":
            confidence = maximum_no_bottle_score
        elif prediction == "normal":
            confidence = maximum_normal_score
        else:
            confidence = max(
                maximum_no_bottle_score,
                maximum_normal_score,
                maximum_anomaly_score,
            )
        return {
            "prediction": prediction,
            "confidence": float(confidence),
            "maximum_entry_score": maximum_entry_score,
            "maximum_fall_score": maximum_fall_score,
            "detection": None,
        }

    first_detection = min(
        detection_candidates,
        key=lambda candidate: candidate["estimated_timestamp_utc"],
    )
    return {
        "prediction": first_detection["detected_class"],
        "confidence": first_detection["detected_score"],
        "maximum_entry_score": maximum_entry_score,
        "maximum_fall_score": maximum_fall_score,
        "detection": first_detection,
    }

def extract_detection_frame(
    video_path,
    frame_index,
    rotate_180,
    destination_path,
):
    """Extract the selected source frame and apply the inference orientation."""

    video_path = Path(video_path)
    if not video_path.is_file():
        raise FileNotFoundError(f"Saved classified video is missing: {video_path}")

    capture = cv2.VideoCapture(str(video_path))
    if not capture.isOpened():
        capture.release()
        raise RuntimeError(f"Could not open saved classified video: {video_path}")

    try:
        capture.set(cv2.CAP_PROP_POS_FRAMES, int(frame_index))
        success, frame = capture.read()
    finally:
        capture.release()

    if not success:
        raise RuntimeError(
            f"Could not extract frame {frame_index} from {video_path.name}"
        )
    should_rotate = (
        rotate_180 is True
        or str(rotate_180).strip().casefold() in {"true", "1", "yes"}
    )
    if should_rotate:
        frame = cv2.rotate(frame, cv2.ROTATE_180)

    destination_path = Path(destination_path)
    destination_path.parent.mkdir(parents=True, exist_ok=True)
    if not cv2.imwrite(str(destination_path), frame):
        raise RuntimeError(f"Could not save evidence frame: {destination_path}")
    return destination_path


def classify_stoppage_events(
    events,
    event_window_clips,
    frame_results,
    clip_results,
    config,
):
    """Classify every PLC stop from the first non-normal event-window frame."""

    # Use one stable frame result per S3 key and source frame.
    frame_results = frame_results.drop_duplicates(
        ["s3_key", "frame_index"],
        keep="last",
    ).copy()
    frame_results["estimated_frame_timestamp_utc"] = (
        pd.to_datetime(frame_results["clip_timestamp_utc"], utc=True)
        + pd.to_timedelta(frame_results["frame_time_seconds"], unit="s")
    )

    # Use the latest successful clip row to locate the retained local video.
    clip_results = clip_results.copy()
    clip_results["result_row"] = np.arange(len(clip_results))
    latest_successful_clips = (
        clip_results.loc[
            clip_results["status"].astype(str).str.casefold() == "success"
        ]
        .sort_values("result_row")
        .drop_duplicates("s3_key", keep="last")
    )
    clip_lookup = latest_successful_clips.set_index("s3_key").to_dict(
        orient="index"
    )

    output_rows = []
    event_frame_root = config.output_dir / "stoppage_event_frames"
    event_window_delta = pd.Timedelta(minutes=config.event_window_minutes)

    for _, event in events.iterrows():
        event_id = str(event["event_id"])
        event_time_utc = pd.Timestamp(event["event_time_utc"])
        event_window_start = event_time_utc - event_window_delta
        event_window_end = event_time_utc + event_window_delta

        mapped_rows = event_window_clips.loc[
            event_window_clips["event_id"].astype(str) == event_id
        ]
        permitted_keys = set(mapped_rows["s3_key"].astype(str))
        event_frames = frame_results.loc[
            frame_results["s3_key"].astype(str).isin(permitted_keys)
            & (
                frame_results["estimated_frame_timestamp_utc"]
                >= event_window_start
            )
            & (
                frame_results["estimated_frame_timestamp_utc"]
                <= event_window_end
            )
        ].copy()

        row = event.to_dict()
        row.update({
            "event_window_start_utc": event_window_start.isoformat(),
            "event_window_end_utc": event_window_end.isoformat(),
            "window_clip_count": len(permitted_keys),
            "scored_frame_count": len(event_frames),
            "classification_status": "success",
            "stoppage_class": "",
            "confidence": np.nan,
            "maximum_fallen_before_entry_score": np.nan,
            "maximum_fallen_in_view_score": np.nan,
            "first_non_normal_s3_key": "",
            "first_non_normal_frame_index": np.nan,
            "first_non_normal_frame_time_seconds": np.nan,
            "first_non_normal_timestamp_utc": "",
            "first_non_normal_timestamp_local": "",
            "first_non_normal_offset_from_stop_seconds": np.nan,
            "first_non_normal_probability": np.nan,
            "first_non_normal_frame_class_probability": np.nan,
            "first_non_normal_window_start_utc": "",
            "first_non_normal_window_end_utc": "",
            "classified_video_path": "",
            "extracted_frame_path": "",
            "frame_extraction_error": "",
        })

        if not permitted_keys:
            row["classification_status"] = "no_s3_clips_in_window"
            output_rows.append(row)
            continue
        if event_frames.empty:
            row["classification_status"] = "no_classified_frames_in_window"
            output_rows.append(row)
            continue

        decision = find_first_non_normal_detection(
            event_frame_rows=event_frames,
            config=config,
        )
        row["stoppage_class"] = decision["prediction"]
        row["confidence"] = decision["confidence"]
        row["maximum_fallen_before_entry_score"] = decision[
            "maximum_entry_score"
        ]
        row["maximum_fallen_in_view_score"] = decision[
            "maximum_fall_score"
        ]

        detection = decision["detection"]
        if detection is None:
            output_rows.append(row)
            continue

        selected_frame = detection["selected_frame"]
        detected_timestamp_utc = pd.Timestamp(
            detection["estimated_timestamp_utc"]
        )
        s3_key = str(selected_frame["s3_key"])
        frame_index = int(selected_frame["frame_index"])
        clip_record = clip_lookup.get(s3_key, {})
        saved_video_path = str(clip_record.get("saved_video_path", ""))

        row.update({
            "first_non_normal_s3_key": s3_key,
            "first_non_normal_frame_index": frame_index,
            "first_non_normal_frame_time_seconds": float(
                selected_frame["frame_time_seconds"]
            ),
            "first_non_normal_timestamp_utc": detected_timestamp_utc.isoformat(),
            "first_non_normal_timestamp_local": detected_timestamp_utc.tz_convert(
                config.event_timezone
            ).isoformat(),
            "first_non_normal_offset_from_stop_seconds": (
                detected_timestamp_utc - event_time_utc
            ).total_seconds(),
            "first_non_normal_probability": detection["detected_score"],
            "first_non_normal_frame_class_probability": detection[
                "selected_frame_class_probability"
            ],
            "first_non_normal_window_start_utc": pd.Timestamp(
                detection["rolling_window_start_utc"]
            ).isoformat(),
            "first_non_normal_window_end_utc": pd.Timestamp(
                detection["rolling_window_end_utc"]
            ).isoformat(),
            "classified_video_path": saved_video_path,
        })

        timestamp_for_filename = detected_timestamp_utc.strftime(
            "%Y%m%dT%H%M%S_%fZ"
        )
        frame_filename = (
            f"{event_id}_{decision['prediction']}_"
            f"{timestamp_for_filename}_frame_{frame_index:06d}.jpg"
        )
        frame_path = (
            event_frame_root
            / decision["prediction"]
            / frame_filename
        )

        try:
            extracted_path = extract_detection_frame(
                video_path=saved_video_path,
                frame_index=frame_index,
                rotate_180=clip_record.get("rotated_180", False),
                destination_path=frame_path,
            )
            row["extracted_frame_path"] = str(extracted_path)
        except Exception as exception:
            row["frame_extraction_error"] = str(exception)

        output_rows.append(row)

    return pd.DataFrame(output_rows)


In [ ]:
frame_results_path = CFG.output_dir / "frame_classification_results.csv"
clip_results_path = CFG.output_dir / "clip_classification_results.csv"
stoppage_results_path = CFG.output_dir / "stoppage_event_classification.csv"

if not frame_results_path.exists() or not clip_results_path.exists():
    print("Run the S3 classification section before classifying stoppages.")
else:
    frame_results = pd.read_csv(frame_results_path)
    clip_results = pd.read_csv(clip_results_path)
    stoppage_results = classify_stoppage_events(
        events=events,
        event_window_clips=event_window_clips,
        frame_results=frame_results,
        clip_results=clip_results,
        config=CFG,
    )
    stoppage_results.to_csv(stoppage_results_path, index=False)

    print(f"Stoppages classified: {len(stoppage_results):,}")
    print()
    print("Class totals:")
    print(stoppage_results["stoppage_class"].fillna("not_scored").replace("", "not_scored").value_counts().to_string())
    print()
    print(f"Saved to: {stoppage_results_path}")

    display_columns = [
        "event_id",
        "Event Time",
        "stoppage_class",
        "confidence",
        "first_non_normal_timestamp_utc",
        "first_non_normal_offset_from_stop_seconds",
        "first_non_normal_s3_key",
        "first_non_normal_frame_index",
        "extracted_frame_path",
        "classification_status",
    ]
    display(stoppage_results[display_columns].head(50))

---

## 8. Score stoppage detection performance

A stoppage is detected when either anomaly class is the first non-normal event in its configured event window. Because every source event is positive, this chapter reports recall and missed events rather than precision or F1.

In [ ]:
if not stoppage_results_path.exists():
    print("No stoppage results exist yet.")
else:
    if "stoppage_results" not in globals():
        stoppage_results = pd.read_csv(stoppage_results_path)

    successfully_scored = (
        stoppage_results["classification_status"] == "success"
    )
    detected_stoppages = successfully_scored & stoppage_results[
        "stoppage_class"
    ].isin(ANOMALY_CLASSES)
    events_with_window_clips = stoppage_results["window_clip_count"] > 0

    total_events = len(stoppage_results)
    scored_count = int(successfully_scored.sum())
    detected_count = int(detected_stoppages.sum())
    events_with_clips_count = int(events_with_window_clips.sum())

    def safe_ratio(numerator, denominator):
        return float(numerator / denominator) if denominator else np.nan

    performance_summary = pd.DataFrame([{
        "total_positive_stoppages": total_events,
        "events_with_s3_window_clips": events_with_clips_count,
        "successfully_scored_stoppages": scored_count,
        "detected_stoppages": detected_count,
        "window_clip_coverage": safe_ratio(events_with_clips_count, total_events),
        "classification_coverage": safe_ratio(scored_count, total_events),
        "recall_on_scored_stoppages": safe_ratio(detected_count, scored_count),
        "end_to_end_recall": safe_ratio(detected_count, total_events),
        "fallen_threshold": CFG.fallen_threshold,
        "falling_threshold": CFG.falling_threshold,
        "event_window_minutes_each_side": CFG.event_window_minutes,
    }])

    performance_summary_path = CFG.output_dir / "stoppage_detection_performance.csv"
    performance_summary.to_csv(performance_summary_path, index=False)
    display(performance_summary.style.format({
        "window_clip_coverage": "{:.1%}",
        "classification_coverage": "{:.1%}",
        "recall_on_scored_stoppages": "{:.1%}",
        "end_to_end_recall": "{:.1%}",
    }))
    print(f"Performance summary: {performance_summary_path}")

### Threshold sensitivity

This uses each stoppage's maximum rolling score across the two anomaly classes to show positive-event recall at different shared thresholds. Normal/control windows are still needed before selecting production thresholds.

In [ ]:
if not stoppage_results_path.exists():
    print("No stoppage results are available for a threshold sweep.")
else:
    scored_stoppages = stoppage_results.loc[
        stoppage_results["classification_status"] == "success"
    ].copy()
    scored_stoppages["maximum_anomaly_score"] = scored_stoppages[
        [
            "maximum_fallen_before_entry_score",
            "maximum_fallen_in_view_score",
        ]
    ].max(axis=1)

    threshold_rows = []
    for threshold in np.round(np.arange(0.50, 0.96, 0.05), 2):
        detected_count = int(
            (scored_stoppages["maximum_anomaly_score"] >= threshold).sum()
        )
        threshold_rows.append({
            "threshold": threshold,
            "scored_positive_stoppages": len(scored_stoppages),
            "detected_positive_stoppages": detected_count,
            "recall": safe_ratio(detected_count, len(scored_stoppages)),
        })

    threshold_sweep = pd.DataFrame(threshold_rows)
    threshold_sweep_path = CFG.output_dir / "stoppage_threshold_recall_sweep.csv"
    threshold_sweep.to_csv(threshold_sweep_path, index=False)
    display(threshold_sweep.style.format({"threshold": "{:.2f}", "recall": "{:.1%}"}))

    axis = threshold_sweep.plot(
        x="threshold",
        y="recall",
        marker="o",
        ylim=(0, 1.05),
        grid=True,
        legend=False,
        figsize=(8, 4),
        title="Stoppage recall versus anomaly threshold",
    )
    axis.set_ylabel("Recall on scored positive stoppages")
    plt.show()
    print(f"Threshold sweep: {threshold_sweep_path}")

### Stoppage detection table

A simple comparison of the Hartford extract time and LoggerBox detection time.

In [ ]:
if not stoppage_results_path.exists():
    print("No stoppage results are available.")
else:
    if "stoppage_results" not in globals():
        stoppage_results = pd.read_csv(stoppage_results_path)

    stoppage_table = stoppage_results[
        ["Event Time", "first_non_normal_timestamp_utc", "stoppage_class"]
    ].copy()
    stoppage_table.columns = [
        "Hartford extract time",
        "LoggerBox detect time UTC",
        "Reason",
    ]

    stoppage_table_path = CFG.output_dir / "stoppage_detection_table.csv"
    stoppage_table.to_csv(stoppage_table_path, index=False)

    display(stoppage_table.fillna(""))
    print(f"Stoppage detection table: {stoppage_table_path}")